# Bab 11 dan 12 — Evaluasi dan studi kasus

Memerlukan output/ (notebook 01) dan output/prediksi/ serta hasil_*.csv dari notebook 03. Sel persiapan memuat ulang data dan fungsi. Kode 11.8 (sepuluh seed) memakan sekitar 15 sampai 30 menit pada mesin dua inti.

## Persiapan

In [ ]:
from sklearn.datasets import make_classification
import os, json, time, platform, warnings
from collections import Counter
import numpy as np, pandas as pd
import sklearn, imblearn, xgboost
from sklearn.pipeline import Pipeline as SkPipeline
from imblearn.pipeline import Pipeline            # pipeline yang menerima sampler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
os.makedirs("output/prediksi", exist_ok=True)

# Data dari Bab 8: X_train/X_test asli (tanpa balancing), label multikelas dan biner
X_train = pd.read_csv("output/X_train.csv");  X_test = pd.read_csv("output/X_test.csv")
y_train = pd.read_csv("output/y_train.csv")["y"].to_numpy()
y_test = pd.read_csv("output/y_test.csv")["y"].to_numpy()
yb_train = pd.read_csv("output/y_train_biner.csv")["y_biner"].to_numpy()
yb_test = pd.read_csv("output/y_test_biner.csv")["y_biner"].to_numpy()
nama_kelas = pd.read_csv("output/pemetaan_label.csv")["label"].tolist()

kolom_num = X_train.select_dtypes(include="number").columns.tolist()
kolom_kat = [c for c in X_train.columns if c not in kolom_num]
print("Latih:", X_train.shape, "| Uji:", X_test.shape)
print("Fitur numerik:", len(kolom_num), "| kategorikal (tidak dipakai di bab ini):", kolom_kat)

def praproses():
    """Imputasi median dan scaling untuk fitur numerik. Dipasang di dalam pipeline,
    sehingga statistiknya hanya dipelajari dari data latih (atau lipatan latih)."""
    return ColumnTransformer(
        [("num", SkPipeline([("imputer", SimpleImputer(strategy="median")),
                             ("scaler", StandardScaler())]), kolom_num)],
        remainder="drop")                      # kolom kategorikal dibuang (subbab 10.1)

from sklearn.ensemble import RandomForestClassifier

def model_rf(class_weight=None):
    return RandomForestClassifier(n_estimators=100, max_depth=None,
                                  class_weight=class_weight,
                                  n_jobs=-1, random_state=RANDOM_STATE)

p = model_rf().get_params()
print("Random Forest:", p["n_estimators"], "pohon | max_depth =", p["max_depth"], "| class_weight =", p["class_weight"])

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

recall_kelas = []          # recall per kelas tiap konfigurasi (dipakai Bab 11)

def gmean(y_true, y_pred, labels):
    """Rata-rata geometrik recall tiap kelas. Bernilai 0 bila ada kelas yang tidak terdeteksi."""
    r = recall_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    return float(np.prod(r) ** (1 / len(r)))

def evaluasi(skenario, model_nama, sampler_nama, pipa, X_tr, y_tr, X_te, y_te, fit_kwargs=None):
    """Latih pipeline pada data latih, nilai pada data uji ASLI; catat metrik dan waktu."""
    labels = np.unique(y_te)
    t0 = time.perf_counter(); pipa.fit(X_tr, y_tr, **(fit_kwargs or {}))
    waktu_fit = time.perf_counter() - t0                   # termasuk praproses dan sampling
    tp = []
    for _ in range(5):                                      # ulangi agar waktu prediksi stabil
        t0 = time.perf_counter(); y_pred = pipa.predict(X_te); tp.append(time.perf_counter() - t0)
    proba = pipa.predict_proba(X_te).astype("float32")
    np.savez_compressed(f"output/prediksi/{skenario}_{model_nama}_{sampler_nama}.npz".replace(" ", "_"),
                        y_pred=y_pred, proba=proba)
    for k, r in zip(labels, recall_score(y_te, y_pred, labels=labels, average=None, zero_division=0)):
        recall_kelas.append({"skenario": skenario, "model": model_nama, "sampler": sampler_nama,
                             "kelas": int(k), "recall": float(r)})
    return {"skenario": skenario, "model": model_nama, "sampler": sampler_nama, "status": "ok",
            "akurasi": accuracy_score(y_te, y_pred),
            "presisi_macro": precision_score(y_te, y_pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_te, y_pred, average="macro", zero_division=0),
            "f1_macro": f1_score(y_te, y_pred, average="macro", zero_division=0),
            "gmean": gmean(y_te, y_pred, labels),
            "waktu_fit_dtk": waktu_fit, "waktu_prediksi_dtk": float(np.median(tp))}

In [ ]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

def model_xgb():
    return XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                         tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE)

## Kode 11.1 (kode111)

In [ ]:
# Contoh ILUSTRASI (angka rekaan, bukan hasil penelitian): 3 kelas, baris = kelas sebenarnya, kolom = prediksi
import numpy as np
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
nama3 = ["Normal", "Serangan A", "Serangan B"]
cm = np.array([[940, 40, 20],
               [ 10, 80, 10],
               [  5,  5, 40]])
y_true = np.repeat([0, 1, 2], cm.sum(axis=1))                  # susun label dari matriks agar fungsi sklearn bisa dipakai
y_pred = np.concatenate([np.repeat([0, 1, 2], cm[i]) for i in range(3)])
assert (confusion_matrix(y_true, y_pred) == cm).all()

TP = np.diag(cm); FP = cm.sum(axis=0) - TP; FN = cm.sum(axis=1) - TP; TN = cm.sum() - TP - FP - FN
presisi, recall, f1, dukungan = precision_recall_fscore_support(y_true, y_pred, zero_division=0)
spesifisitas = TN / (TN + FP); fpr = FP / (FP + TN); fnr = FN / (FN + TP)
for i, n in enumerate(nama3):
    print(f"{n:11s} TP={TP[i]:4d} FP={FP[i]:3d} FN={FN[i]:3d} TN={TN[i]:4d} | presisi={presisi[i]:.4f} recall={recall[i]:.4f} "
          f"F1={f1[i]:.4f} spes={spesifisitas[i]:.4f} FPR={fpr[i]:.4f} FNR={fnr[i]:.4f}")
print("Akurasi    :", round(TP.sum() / cm.sum(), 4))
print("F1 macro   :", round(f1.mean(), 4), "| recall macro:", round(recall.mean(), 4))
print("G-Mean (3 kelas)       :", round(float(np.prod(recall) ** (1 / 3)), 4))
print("Weighted F1            :", round(float((f1 * dukungan).sum() / dukungan.sum()), 4))

## Kode 11.2 (kode112)

In [ ]:
# Metrik lengkap untuk satu konfigurasi (XGB tanpa sampler, multikelas), dari prediksi yang tersimpan di Bab 10
from sklearn.metrics import (confusion_matrix, classification_report, roc_auc_score,
                             average_precision_score)
from imblearn.metrics import classification_report_imbalanced, geometric_mean_score
from sklearn.preprocessing import label_binarize

d = np.load("output/prediksi/multikelas_XGB_Baseline.npz")
y_pred, proba = d["y_pred"], d["proba"].astype("float64")
proba = proba / proba.sum(axis=1, keepdims=True)               # float32 -> jumlah baris tepat 1
labels = np.arange(len(nama_kelas))

print(classification_report(y_test, y_pred, labels=labels, target_names=nama_kelas, digits=4, zero_division=0))
print("G-Mean (imblearn)      :", round(geometric_mean_score(y_test, y_pred, average="multiclass"), 4))
print("ROC-AUC OvR macro      :", round(roc_auc_score(y_test, proba, multi_class="ovr", average="macro"), 4))
Y = label_binarize(y_test, classes=labels)
print("PR-AUC (AP) macro      :", round(float(np.mean([average_precision_score(Y[:, k], proba[:, k]) for k in labels])), 4))

## Kode 11.3 (kode113)

In [ ]:
import matplotlib.pyplot as plt
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_norm = cm / cm.sum(axis=1, keepdims=True)                   # tiap baris dijumlahkan menjadi 1 (= recall pada diagonal)
fig, ax = plt.subplots(figsize=(9.5, 8))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
for i in labels:
    for j in labels:
        if cm[i, j] > 0:
            ax.text(j, i, f"{cm[i, j]:,}".replace(",", "."), ha="center", va="center", fontsize=7,
                    color="white" if cm_norm[i, j] > 0.5 else "black")
ax.set_xticks(labels); ax.set_xticklabels(nama_kelas, rotation=60, ha="right", fontsize=7)
ax.set_yticks(labels); ax.set_yticklabels(nama_kelas, fontsize=7)
ax.set_xlabel("Kelas prediksi"); ax.set_ylabel("Kelas sebenarnya")
fig.colorbar(im, ax=ax, fraction=0.04, label="proporsi per baris")
plt.tight_layout(); plt.savefig("output/g_confusion_matrix.png", dpi=200); plt.show()
print("Sel di luar diagonal yang tidak nol:", int(((cm > 0) & ~np.eye(len(labels), dtype=bool)).sum()))

## Kode 11.4 (kode114)

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, auc
pilih = ["DOS_SYN_Hping", "Wipro_bulb", "Metasploit_Brute_Force_SSH", "NMAP_FIN_SCAN"]
warna = ["#1f77b4", "#E07B39", "#2E9C93", "#9467bd"]
tabel_auc = []
fig_roc, a1 = plt.subplots(figsize=(5.2, 4.4)); fig_pr, a2 = plt.subplots(figsize=(5.2, 4.4))
for n, w in zip(pilih, warna):
    k = nama_kelas.index(n)
    fpr_, tpr_, _ = roc_curve(Y[:, k], proba[:, k]); pr_, rc_, _ = precision_recall_curve(Y[:, k], proba[:, k])
    a1.plot(fpr_, tpr_, color=w, label=n); a2.plot(rc_, pr_, color=w, label=n)
    tabel_auc.append({"kelas": n, "sampel_uji": int(Y[:, k].sum()), "ROC_AUC": auc(fpr_, tpr_),
                      "PR_AUC": average_precision_score(Y[:, k], proba[:, k])})
a1.plot([0, 1], [0, 1], "k:", lw=0.8); a1.set_xlabel("False Positive Rate"); a1.set_ylabel("True Positive Rate")
a2.set_xlabel("Recall"); a2.set_ylabel("Presisi")
for a in (a1, a2): a.legend(fontsize=7, loc="lower right" if a is a1 else "lower left"); a.grid(alpha=0.25)
fig_roc.tight_layout(); fig_roc.savefig("output/g_roc.png", dpi=200)
fig_pr.tight_layout(); fig_pr.savefig("output/g_pr.png", dpi=200)
tabel_auc = pd.DataFrame(tabel_auc); tabel_auc.to_csv("output/auc_kelas_terpilih.csv", index=False)
print(tabel_auc.round(4).to_string(index=False))

## Kode 11.5 (kode115)

In [ ]:
# Laporan per kelas: F1 tiap kelas untuk empat konfigurasi (heatmap) dan tabel presisi/recall/F1 untuk satu konfigurasi
from sklearn.metrics import precision_recall_fscore_support
konfig = [("RF", "Baseline"), ("XGB", "Baseline"), ("XGB", "Borderline-SMOTE"), ("XGB", "SMOTE-ENN")]
f1_kelas = {}
for m, s in konfig:
    yp = np.load(f"output/prediksi/multikelas_{m}_{s}.npz")["y_pred"]
    f1_kelas[f"{m} {s}"] = precision_recall_fscore_support(y_test, yp, labels=labels, zero_division=0)[2]
f1_df = pd.DataFrame(f1_kelas, index=nama_kelas)
f1_df["sampel_uji"] = np.bincount(y_test, minlength=len(labels))
f1_df.round(4).to_csv("output/f1_per_kelas.csv")
print(f1_df.round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 4.8))
nilai = f1_df.drop(columns="sampel_uji").to_numpy()
im = ax.imshow(nilai, cmap="Blues", vmin=0.8, vmax=1.0, aspect="auto")
for i in range(nilai.shape[0]):
    for j in range(nilai.shape[1]):
        ax.text(j, i, f"{nilai[i, j]:.3f}".replace(".", ","), ha="center", va="center", fontsize=7,
                color="white" if nilai[i, j] > 0.94 else "black")
ax.set_xticks(range(nilai.shape[1])); ax.set_xticklabels(f1_df.columns[:-1], fontsize=7)
ax.set_yticks(range(len(nama_kelas))); ax.set_yticklabels([f"{n} ({c})" for n, c in zip(nama_kelas, f1_df['sampel_uji'])], fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.04, label="F1 (skala 0,8 sampai 1,0)")
plt.tight_layout(); plt.savefig("output/g_f1_per_kelas.png", dpi=200); plt.show()

## Kode 11.6 (kode116)

In [ ]:
# Analisis kesalahan: (a) pasangan kelas yang paling sering tertukar (XGB baseline, multikelas)
#                     (b) alarm palsu dan serangan lolos pada skenario biner
asal, tujuan = np.where(~np.eye(len(labels), dtype=bool) & (cm > 0))
pasangan = sorted(zip(cm[asal, tujuan], asal, tujuan), reverse=True)[:8]
for n, i, j in pasangan:
    print(f"{n:3d} sampel {nama_kelas[i]} -> diprediksi {nama_kelas[j]}")

print("\nBiner (0 = normal, 1 = serangan); FP = alarm palsu, FN = serangan lolos")
baris = []
for m in ("RF", "XGB"):
    for s in ("Baseline", "RUS", "SMOTE", "SMOTE-ENN"):
        yp = np.load(f"output/prediksi/biner_{m}_{s}.npz")["y_pred"]
        tn, fp, fn, tp = confusion_matrix(yb_test, yp).ravel()
        baris.append({"model": m, "sampler": s, "FP": fp, "FN": fn, "FPR": fp / (fp + tn), "FNR": fn / (fn + tp)})
galat = pd.DataFrame(baris); galat.to_csv("output/galat_biner.csv", index=False)
print("Jumlah normal pada data uji:", int((yb_test == 0).sum()), "| serangan:", int((yb_test == 1).sum()))
print(galat.round(5).to_string(index=False))

## Kode 11.7 (kode117)

In [ ]:
# Sumber daya model baseline multikelas: ukuran model, memori proses, dan latensi prediksi satu baris
import psutil, pickle, gc
proses = psutil.Process()
hasil_sd = []
for nama, buat in (("RF", model_rf), ("XGB", model_xgb)):
    gc.collect(); rss0 = proses.memory_info().rss
    pipa = Pipeline([("praproses", praproses()), ("model", buat())]).fit(X_train, y_train)
    rss1 = proses.memory_info().rss
    ukuran_mb = len(pickle.dumps(pipa)) / 1e6                  # ukuran model terserialisasi
    satu = X_test.iloc[[0]]; lat = []
    for i in range(200):                                       # 200 prediksi satu baris
        t0 = time.perf_counter(); pipa.predict(X_test.iloc[[i]]); lat.append((time.perf_counter() - t0) * 1000)
    t0 = time.perf_counter(); pipa.predict(X_test); batch = (time.perf_counter() - t0) / len(X_test) * 1000
    hasil_sd.append({"model": nama, "ukuran_pickle_MB": ukuran_mb, "tambahan_RSS_MB": (rss1 - rss0) / 1e6,
                     "latensi_1_baris_ms_median": float(np.median(lat)), "latensi_per_baris_batch_ms": batch})
sd = pd.DataFrame(hasil_sd); sd.to_csv("output/sumber_daya.csv", index=False)
print(sd.round(3).to_string(index=False))

## Kode 11.8 (kode118)

In [ ]:
# Uji konsistensi: ulangi pembagian data dan pelatihan dengan 10 seed (multikelas); simpan tiap hasil segera
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE, BorderlineSMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, recall_score
import csv, time

X_all = pd.concat([X_train, X_test], ignore_index=True)           # data bersih Bab 8 (setelah duplikat dibuang)
y_all = np.concatenate([y_train, y_test])
SEEDS = list(range(42, 52))
SAMPLER_RF = ["Baseline", "RUS", "ROS", "SMOTE", "Borderline-SMOTE"]
SAMPLER_XGB = ["Baseline", "RUS", "SMOTE", "Borderline-SMOTE"]     # hybrid dan ADASYN tidak diulang (lihat teks)
kelas_langka = [nama_kelas.index("NMAP_FIN_SCAN"), nama_kelas.index("Metasploit_Brute_Force_SSH")]

def buat_sampler(nama, jumlah, seed):
    batas = {k: min(v, 2000) for k, v in jumlah.items()}
    target = {k: max(v, 2000) for k, v in jumlah.items()}
    K = max(1, min(5, min(jumlah.values()) - 1))
    return {"Baseline": None,
            "RUS": RandomUnderSampler(sampling_strategy=batas, random_state=seed),
            "ROS": RandomOverSampler(sampling_strategy=target, random_state=seed),
            "SMOTE": SMOTE(sampling_strategy=target, k_neighbors=K, random_state=seed),
            "Borderline-SMOTE": BorderlineSMOTE(sampling_strategy=target, k_neighbors=K, random_state=seed)}[nama]

berkas = "output/hasil_seed.csv"
kolom_csv = ["seed", "model", "sampler", "f1_macro", "gmean", "recall_FIN", "recall_Metasploit", "waktu_fit_dtk"]
with open(berkas, "w", newline="") as f:
    csv.writer(f).writerow(kolom_csv)
for seed in SEEDS:
    RANDOM_STATE = seed                                              # dibaca oleh model_rf dan model_xgb
    Xa, Xb, ya, yb = train_test_split(X_all, y_all, test_size=0.30, stratify=y_all, random_state=seed)
    jumlah = Counter(ya)
    for model_nama, buat, daftar in (("RF", model_rf, SAMPLER_RF), ("XGB", model_xgb, SAMPLER_XGB)):
        for s_nama in daftar:
            s = buat_sampler(s_nama, jumlah, seed)
            langkah = [("praproses", praproses())] + ([("sampler", s)] if s is not None else []) + [("model", buat())]
            t0 = time.perf_counter(); pipa = Pipeline(langkah).fit(Xa, ya); wf = time.perf_counter() - t0
            yp = pipa.predict(Xb)
            r = recall_score(yb, yp, labels=kelas_langka, average=None, zero_division=0)
            with open(berkas, "a", newline="") as f:
                csv.writer(f).writerow([seed, model_nama, s_nama, f1_score(yb, yp, average="macro"),
                                        gmean(yb, yp, np.unique(yb)), r[0], r[1], wf])
    print("seed", seed, "selesai", flush=True)
hasil_seed = pd.read_csv(berkas)
print(hasil_seed.groupby(["model", "sampler"])[["f1_macro", "gmean"]].agg(["mean", "std"]).round(4).to_string())

## Kode 11.9 (kode119)

In [ ]:
# Konsistensi antarseed: rata-rata, simpangan baku, uji Wilcoxon berpasangan terhadap Baseline, dan uji Friedman
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, friedmanchisquare
hs = pd.read_csv("output/hasil_seed.csv")
ringkas = hs.groupby(["model", "sampler"])[["f1_macro", "gmean"]].agg(["mean", "std"]).round(4)
print(ringkas.to_string())

baris = []
for m in ("RF", "XGB"):
    w = hs[hs.model == m].pivot(index="seed", columns="sampler", values="f1_macro")
    for s in [c for c in w.columns if c != "Baseline"]:
        beda = w[s] - w["Baseline"]
        p = wilcoxon(w[s], w["Baseline"]).pvalue
        baris.append({"model": m, "sampler": s, "selisih_rata2": beda.mean(), "menang_dari_baseline": int((beda > 0).sum()),
                      "dari_seed": len(beda), "p_wilcoxon": p})
    fr = friedmanchisquare(*[w[c] for c in w.columns])
    print(m, "Friedman: statistik =", round(fr.statistic, 3), "| p =", round(fr.pvalue, 5))
uji = pd.DataFrame(baris); uji.to_csv("output/uji_statistik.csv", index=False)
print(uji.round(4).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
for a, m in zip(ax, ("RF", "XGB")):
    w = hs[hs.model == m].pivot(index="seed", columns="sampler", values="f1_macro")
    a.boxplot([w[c] for c in w.columns], tick_labels=list(w.columns), showmeans=True)
    a.set_title(m); a.tick_params(axis="x", labelrotation=25, labelsize=8); a.grid(axis="y", alpha=0.25)
ax[0].set_ylabel("F1 macro (data uji tiap seed)")
plt.tight_layout(); plt.savefig("output/g_seed_boxplot.png", dpi=200); plt.show()

## Kode 11.10 (kode1110)

In [ ]:
# Mengapa recall kelas langka identik pada semua sampler? Bandingkan sampel uji yang salah pada seed 45 (RF, multikelas)
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
X_all = pd.concat([X_train, X_test], ignore_index=True); y_all = np.concatenate([y_train, y_test])
seed = 45; RANDOM_STATE = seed
Xa, Xb, ya, yb = train_test_split(X_all, y_all, test_size=0.30, stratify=y_all, random_state=seed)
jum = Counter(ya); tg = {k: max(v, 2000) for k, v in jum.items()}
res = {}
for nm, s in (("base", None), ("smote", SMOTE(sampling_strategy=tg, k_neighbors=5, random_state=seed))):
    pipa = Pipeline([("praproses", praproses())] + ([("s", s)] if s else []) + [("model", model_rf())]).fit(Xa, ya)
    res[nm] = pipa.predict(Xb)
for nama in ("NMAP_FIN_SCAN", "Metasploit_Brute_Force_SSH"):
    k = nama_kelas.index(nama); idx = np.where(yb == k)[0]
    for nm in res:
        salah = [(int(i), nama_kelas[res[nm][i]]) for i in idx if res[nm][i] != k]
        print(nama, nm, "salah:", salah)

## Kode 12.1 (kode121)

In [ ]:
# Tabel ringkasan studi kasus: metrik multikelas (termasuk ROC-AUC), FPR/FNR biner, ukuran data latih, waktu
from sklearn.metrics import roc_auc_score, confusion_matrix
ukuran_latih = {"Baseline": 82545, "RUS": 13888, "ROS": 92657, "SMOTE": 92657, "Borderline-SMOTE": 89484,
                "SMOTE-Tomek": 92597, "SMOTE-ENN": 92101, "class_weight": 82545, "sample_weight": 82545}   # dari Bab 9
multi = pd.read_csv("output/hasil_multikelas.csv"); multi = multi[multi["status"] == "ok"]
biner = pd.read_csv("output/hasil_biner.csv"); biner = biner[biner["status"] == "ok"].set_index(["model", "sampler"])
baris = []
for r in multi.itertuples():
    nama = f"output/prediksi/multikelas_{r.model}_{r.sampler}.npz".replace(" ", "_")
    proba = np.load(nama)["proba"].astype("float64"); proba /= proba.sum(axis=1, keepdims=True)
    auc = roc_auc_score(y_test, proba, multi_class="ovr", average="macro")
    yp = np.load(f"output/prediksi/biner_{r.model}_{r.sampler}.npz")["y_pred"]
    tn, fp, fn, tp = confusion_matrix(yb_test, yp).ravel()
    baris.append({"model": r.model, "sampler": r.sampler, "f1_macro": r.f1_macro, "gmean": r.gmean, "roc_auc_macro": auc,
                  "FPR_biner": fp / (fp + tn), "FNR_biner": fn / (fn + tp),
                  "ukuran_latih_multi": ukuran_latih[r.sampler], "fit_multi_dtk": r.waktu_fit_dtk,
                  "fit_biner_dtk": biner.loc[(r.model, r.sampler), "waktu_fit_dtk"]})
studi = pd.DataFrame(baris); studi.to_csv("output/ringkasan_studi_kasus.csv", index=False)
print(studi[["model", "sampler", "f1_macro", "gmean", "roc_auc_macro", "FPR_biner", "FNR_biner", "fit_multi_dtk"]].round(4).to_string(index=False))